# A7 · XGBoost: a second opinion, compared fairly

**Where we are**

```
4 Baseline → B3: 1.258 on June (q = 0.8)
5 Train    → A6 LightGBM: 1.138.  HERE (A7): XGBoost, same data, same rules. Which one goes forward?
6 Evaluate → A8: the chosen model meets the locked test months (Jul–Aug) once
```

**Why train a second model?** One model's score can be luck of its settings. If a different library, trained the same way, lands close by, we trust the result more. And if it's better, we use it.

**A fair comparison means the same everything:** same training rows (Jan–May), same 10 columns, same q values, same early stopping on June, same score. Only the library changes.

**The one real difference, in plain words**

| | How a tree grows | Capped by |
|---|---|---|
| LightGBM | **leaf by leaf**: always split the single box that lowers the score most | `num_leaves = 31` |
| XGBoost | **level by level**: split every box at one depth, then the next depth | `max_depth = 6` (up to 64 boxes) |

The code is written for you this time (`slotpred/train_xgb.py`), because of the deadline. Read it: it mirrors your `train_lgbm.py` line by line.

In [1]:
%load_ext autoreload
%autoreload 2

import time
import numpy as np
import pandas as pd

from slotpred.config import OUT, Q_CANDIDATES
from slotpred.features import make_xy
from slotpred.metrics import evaluate
from slotpred.prepare import load_prepared
from slotpred import train_lgbm, train_xgb

splits, state = load_prepared()
X_train, y_train = make_xy(splits["train"], state)
X_valid, y_valid = make_xy(splits["valid"], state)

## 1 · Train both, timing each

In [2]:
t = time.time(); xgb_models = train_xgb.fit_all(X_train, y_train, X_valid, y_valid); xgb_s = time.time() - t
t = time.time(); lgb_models = train_lgbm.fit_all(X_train, y_train, X_valid, y_valid); lgb_s = time.time() - t
print(f"XGBoost {xgb_s:.1f} s, LightGBM {lgb_s:.1f} s for all three q")
print("trees kept, XGBoost:", {q: m.best_iteration + 1 for q, m in xgb_models.items()})
print("trees kept, LightGBM:", {q: m.best_iteration_ for q, m in lgb_models.items()})

XGBoost 4.4 s, LightGBM 0.7 s for all three q
trees kept, XGBoost: {0.67: 409, 0.75: 541, 0.8: 525}
trees kept, LightGBM: {0.67: 122, 0.75: 107, 0.8: 126}


## 2 · The June leaderboard

In [3]:
rows = []
for q in Q_CANDIDATES:
    rows.append(evaluate(y_valid, lgb_models[q].predict(X_valid), q, "LightGBM"))
    rows.append(evaluate(y_valid, xgb_models[q].predict(X_valid), q, "XGBoost"))
models_board = pd.DataFrame(rows)
models_board[models_board.method == "XGBoost"].to_csv(OUT / "xgb_valid.csv", index=False)

baselines = pd.read_csv(OUT / "baselines_valid.csv")
keep = baselines[baselines.method.str.match("B3|Oracle")]
board = pd.concat([keep, models_board])
cols = ["method", "q", "pinball", "coverage", "booked_on_time", "booked_idle_min"]
board[cols].sort_values(["q", "pinball"]).round(3).reset_index(drop=True)

,method,q,pinball,coverage,booked_on_time,booked_idle_min
0,Oracle (floor),0.67,1.372,0.664,0.843,4.132
1,LightGBM,0.67,1.388,0.663,0.843,4.132
2,XGBoost,0.67,1.392,0.660,0.838,4.054
3,B3 doctor + visit type (backoff to B2),0.67,1.531,0.666,0.848,4.502
4,Oracle (floor),0.75,1.243,0.745,0.890,4.982
5,LightGBM,0.75,1.260,0.746,0.886,4.902
6,XGBoost,0.75,1.266,0.739,0.883,4.880
7,B3 doctor + visit type (backoff to B2),0.75,1.391,0.749,0.872,5.043
8,Oracle (floor),0.80,1.120,0.794,0.914,5.585
9,LightGBM,0.80,1.138,0.795,0.909,5.517


## 3 · Do the two models agree visit by visit?

If two different libraries give almost the same P80 for each visit, both have found the same real pattern.

In [4]:
for q in Q_CANDIDATES:
    a, b = lgb_models[q].predict(X_valid), xgb_models[q].predict(X_valid)
    print(f"q={q}: average difference {np.mean(np.abs(a - b)):.2f} min; booked slots identical for {np.mean(np.ceil(a/5) == np.ceil(b/5)):.0%} of visits")

q=0.67: average difference 0.31 min; booked slots identical for 93% of visits
q=0.75: average difference 0.36 min; booked slots identical for 93% of visits
q=0.8: average difference 0.35 min; booked slots identical for 93% of visits


**Questions**
1. Which model has the lower score at each q? By how much?
2. Which would you take forward to A8, and why? Think about score, coverage, speed, and how much the two disagree.

*Answers:*
LightGBM has the lower score at every q, though the margins are small. booked_on_time is effectively tied: 84.3% vs 83.8% at q=.67, 88.6% vs 88.3% at q=.75, and 90.9% vs 90.6% at q=.80. It also trains all three q values in 0.7 s vs 4.4 s. The models choose the same booked slot for 93% of visits, so they largely agree.